# 04 — SimBench dataset characterization + error decomposition

This notebook has two parts:

**Part I** characterizes the SimBench benchmark itself — dataset composition, the entropy
spectrum from consensus to contested questions, and how much demographic conditioning actually
shifts distributions across groups. These facts are model-independent and establish the
difficulty landscape.

**Part II** takes one run and decomposes *why* its distributions are wrong: splitting each
item's TVD into concentration error (wrong spread) and location error (wrong options), then
attributing score gains to one or the other.

**The decomposition (Part II).** For truth `P` and prediction `Q`, sort both descending and
split the total-variation distance:

```
TVD(P,Q) = concentration_err + location_err
         = ½Σ|sort(P)−sort(Q)|  +  (TVD − that)
```

- **concentration_err** — error in the *profile* (how peaked vs spread the mass is),
  independent of *which* option carries it. *Are we capturing the group's true diversity?*
- **location_err** — extra error from putting well-shaped mass on the *wrong* options.
  *Is the weight in the right place?*

Signed facets add direction: **entropy_gap** = H(Q)−H(P) (`<0` over-sharpened, `>0` too
diffuse), **mode_acc** (did we pick the truth's dominant option).

## Part I: Dataset Characterization

Before diagnosing *why* a system's predictions are wrong, it helps to know the shape
of the benchmark itself. SimBench asks models to predict two kinds of empirical distributions:

- **Population** (`pop`): the aggregate answer for all survey respondents.
- **Grouped** (`grouped`): the answer for a specific demographic segment (e.g. "Finnish women,
  aged 30–49").

This section characterizes both splits — what survey datasets contribute, how contested
vs. consensual the questions are, and how much demographic conditioning actually shifts
the distributions — *independent of any model or run*. These facts determine the difficulty
structure that all later error-decomposition analysis is operating in.

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))

from scrye.config import OUTPUTS_DIR
from scrye.data import load_all
from scrye.splits import make_split, is_required_question, family_key
from scrye.evaluate import build_normalizers
from scrye.decompose import decompose_records, decompose_error, system_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

allr = load_all()
pop_recs = allr["pop"]
grp_recs = allr["grouped"]
full = pop_recs + grp_recs
split = make_split(full, seed=0, unit="question")

print(f"pop records: {len(pop_recs):,}   grouped records: {len(grp_recs):,}   total: {len(full):,}")
print("\nDev/val/test split:")
display(split.summary())

### 1. Dataset Composition

SimBench draws from 20+ survey sources. The **population** split includes all of them;
the **grouped** split only includes the five datasets that published demographic
breakdowns (ESS, Afrobarometer, LatinoBarometro, OpinionQA, ISSP). Question complexity
varies: some items have 2 binary options, others have 10+.

In [ ]:
def _norm_entropy(prob_dict):
    'Normalized Shannon entropy in [0,1] for a {label: prob} dict.'
    p = np.array(list(prob_dict.values()), dtype=float)
    p = p / p.sum()
    p = p[p > 0]
    if len(p) <= 1:
        return 0.0
    h = float(-(p * np.log(p)).sum())
    return h / np.log(len(prob_dict))

def recs_to_df(recs):
    rows = []
    for r in recs:
        rows.append({
            "dataset": r.dataset_name,
            "split": r.split,
            "n_options": r.num_options,
            "is_population": r.is_population,
            "num_grouping_vars": r.num_grouping_vars,
            "group_size": r.group_size,
            "truth_entropy": _norm_entropy(r.human_answer),
        })
    return pd.DataFrame(rows)

df_all = recs_to_df(full)

# Records per dataset × split
tbl = (df_all.groupby(["dataset", "split"])
       .size()
       .unstack(fill_value=0)
       .sort_values("pop", ascending=False))
print("Records per dataset × split:")
display(tbl)

# n_options distribution
print("\nn_options distribution:")
display(df_all["n_options"].value_counts().sort_index().rename("records").to_frame().T)

**Key facts about the dataset:**

- **20 survey datasets** appear in the population split (7,167 records total); only **5** with
  demographic breakdowns appear in the grouped split (6,343 records).
- The five grouped-split datasets — **ESS**, **Afrobarometer**, **LatinoBarometro**,
  **OpinionQA**, **ISSP** — cover beliefs, politics, values, and social attitudes across dozens
  of countries and demographic slices.
- Most items have **4–7 options**, giving enough distributional structure to distinguish
  good from mediocre predictions. A binary item's TVD can only be 0 or 0.5; a 7-option item
  has much richer geometry.
- The three required questions (`trust_president`, `gay_rights`, `internet_use`) are pinned to
  the **test split** and never contact strategy/model selection.

### 2. The Entropy Spectrum: Consensus to Contested

The key variable that determines per-item difficulty is how **genuinely disagreed-upon** the
ground-truth responses are. We measure this with normalized Shannon entropy:

$$H(P) = -\sum_k p_k \log p_k \; / \; \log K$$

where K is the number of options. H = 0 means everyone picked the same answer; H = 1 means
responses are spread perfectly uniformly across all options.

We use tertile thresholds to classify items:

| Regime | What it means for prediction |
|--------|------------------------------|
| **Consensus** (low H) | The "right answer" is clear; the main risk is wrong option placement |
| **Mixed** (mid H) | Moderate spread; some subgroups may pull in different directions |
| **Contested** (high H) | Genuine population-level disagreement; the model must capture real spread |

Contested items are also where **score headroom is largest**: the SimBench normalizer
Z = TVD(P, U) is biggest when P is near-uniform, so each unit of TVD removed buys more score.

In [ ]:
pop_df = df_all[df_all["split"] == "pop"].copy()
grp_df = df_all[df_all["split"] == "grouped"].copy()

# Tertile thresholds on the population-level distribution (used as reference)
p33, p67 = pop_df["truth_entropy"].quantile([1/3, 2/3])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, (sub_df, label, color) in zip(axes, [
    (pop_df, "Population split", "#4C72B0"),
    (grp_df, "Grouped split",    "#DD8452"),
]):
    ax.hist(sub_df["truth_entropy"], bins=40, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(p33, color="black", linestyle="--", lw=1.2)
    ax.axvline(p67, color="black", linestyle="--", lw=1.2)
    ymax = ax.get_ylim()[1]
    ax.text(p33 / 2,           ymax * 0.92, "consensus", ha="center", fontsize=9)
    ax.text((p33 + p67) / 2,   ymax * 0.92, "mixed",     ha="center", fontsize=9)
    ax.text((p67 + 1.0) / 2,   ymax * 0.92, "contested", ha="center", fontsize=9)
    n_contested = (sub_df["truth_entropy"] >= p67).sum()
    pct = 100 * n_contested / len(sub_df)
    ax.set_title(f"{label} (n={len(sub_df):,})\n{pct:.0f}% contested  (H ≥ {p67:.2f})")
    ax.set_xlabel("truth entropy (normalized H)")
    ax.set_ylabel("records")

plt.suptitle("Truth entropy distribution — how contested are SimBench items?", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Population: mean={pop_df['truth_entropy'].mean():.3f}  std={pop_df['truth_entropy'].std():.3f}")
print(f"Grouped:    mean={grp_df['truth_entropy'].mean():.3f}  std={grp_df['truth_entropy'].std():.3f}")
print(f"\nTertile thresholds (from pop): low < {p33:.3f}  |  {p33:.3f}–{p67:.3f}  |  > {p67:.3f}")

In [ ]:
# Show the 3 most consensus and 3 most contested pop questions with their distributions
pop_df_idx = pop_df.copy()  # already has integer index from recs_to_df order

# Sort by entropy; pick extreme tails
pop_sorted = pop_df.sort_values("truth_entropy").reset_index(drop=True)
picks = {
    "Consensus (lowest H)":  pop_sorted.head(3),
    "Contested (highest H)": pop_sorted.tail(3),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
pop_df_global = df_all[df_all["split"] == "pop"].reset_index(drop=True)
for row_i, (label, subset) in enumerate(picks.items()):
    color = "#4C72B0" if row_i == 0 else "#DD8452"
    for col_j, (_, row) in enumerate(subset.iterrows()):
        ax = axes[row_i][col_j]
        # look up matching pop record by entropy + dataset match
        mask = (pop_df_global["truth_entropy"] == row["truth_entropy"]) &                (pop_df_global["dataset"] == row["dataset"])
        orig_idx = pop_df_global[mask].index[0]
        rec = pop_recs[orig_idx]
        opts = list(rec.human_answer.keys())
        probs = [rec.human_answer[o] for o in opts]
        x = np.arange(len(opts))
        ax.bar(x, probs, color=color, alpha=0.85, edgecolor="white")
        ax.set_xticks(x)
        ax.set_xticklabels(opts, fontsize=8)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("proportion")
        q_short = rec.input_template.split("\n")[0][:55]
        ax.set_title(
            f"[{row['dataset']}]  H={row['truth_entropy']:.3f}\n{q_short}…",
            fontsize=8
        )
        if col_j == 0:
            ax.annotate(label, xy=(-0.3, 0.5), xycoords="axes fraction",
                        fontsize=9, fontweight="bold", rotation=90, va="center")

plt.suptitle("Example items at the entropy extremes", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What the entropy distribution tells us:**

- Both splits are **broadly spread** across the full entropy range — roughly one-third in each
  tertile — so there's no "easy dataset" effect where most items are trivially solvable.
- Mean entropy ≈ 0.70 for both splits; the grouped split is **slightly less variable** (std=0.21
  vs 0.26 for pop). Demographic segments can be more internally homogeneous than the whole
  population, pulling some items toward lower entropy.
- The consensus items (low H) are dominated by **near-unanimous answers** — the chart shows
  mass heavily concentrated on one or two options. A system that always picks the majority
  option gets most of these right but earns little score because Z is small.
- The contested items (high H) have mass distributed across many options. **Getting the spread
  right here** — not just picking the mode — is where SimBench's design genuinely challenges
  language models.

### 3. Population vs. Groups: How Much Do Demographics Shift Responses?

The grouped split's value rests on a premise: **demographics change the distribution in
meaningful, learnable ways**. For each question family (one survey question), we can compare
the population entropy to the spread of group-level entropies across all demographic segments.

If groups look like the population (tight scatter around the diagonal), conditioning rarely
changes the shape — the challenge is mostly about option placement. If groups diverge widely
from each other (high inter-group entropy range), there are real demographic patterns a good
model must represent.

In [ ]:
# Build per-family frame: pop entropy vs. distribution of group entropies
pop_by_family = {}
for r in pop_recs:
    fk = family_key(r)
    pop_by_family[fk] = _norm_entropy(r.human_answer)

grp_by_family = {}
for r in grp_recs:
    fk = family_key(r)
    grp_by_family.setdefault(fk, []).append(_norm_entropy(r.human_answer))

family_rows = []
for fk, pop_ent in pop_by_family.items():
    if fk not in grp_by_family:
        continue
    grp_ents = grp_by_family[fk]
    family_rows.append({
        "dataset":            fk[0],
        "pop_entropy":        pop_ent,
        "mean_group_entropy": np.mean(grp_ents),
        "std_group_entropy":  np.std(grp_ents),
        "entropy_range":      max(grp_ents) - min(grp_ents),
        "n_groups":           len(grp_ents),
    })

fam_df = pd.DataFrame(family_rows)
print(f"Question families present in both splits: {len(fam_df):,}")
print("\nPer-dataset summary (n_groups = segments, entropy_range = max−min group H):")
display(fam_df.groupby("dataset")[["n_groups", "entropy_range"]].agg(["mean", "max"]).round(3))

# Scatter: pop entropy vs mean group entropy
fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.cm.tab10.colors
for i, ds in enumerate(sorted(fam_df["dataset"].unique())):
    sub = fam_df[fam_df["dataset"] == ds]
    ax.scatter(sub["pop_entropy"], sub["mean_group_entropy"],
               s=22, alpha=0.55, color=cmap[i % 10], label=ds)
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="groups = population")
ax.set_xlabel("population entropy (H of full survey)")
ax.set_ylabel("mean group entropy (avg H across segments)")
ax.set_title(
    "Pop entropy vs. mean group entropy per question family\n"
    "Above diagonal → groups are more diffuse than the population"
)
ax.legend(fontsize=7.5, ncol=2, loc="upper left", markerscale=1.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histogram of entropy_range (max−min group entropy per family)
axes[0].hist(fam_df["entropy_range"], bins=35, color="#55A868", alpha=0.85, edgecolor="white")
med = fam_df["entropy_range"].median()
axes[0].axvline(med, color="black", linestyle="--", lw=1.2,
                label=f"median = {med:.2f}")
axes[0].set_xlabel("entropy range (max − min group H)")
axes[0].set_ylabel("question families")
axes[0].set_title("Inter-group entropy spread per question family")
axes[0].legend()

# Mean range by dataset (how polarizing is each survey?)
mean_range = (fam_df.groupby("dataset")["entropy_range"]
              .mean()
              .sort_values(ascending=True))
axes[1].barh(mean_range.index, mean_range.values, color="#4C72B0", alpha=0.85, edgecolor="white")
axes[1].set_xlabel("mean entropy range (max − min group H)")
axes[1].set_title("Mean inter-group entropy spread by dataset")

plt.tight_layout()
plt.show()

print("5 question families with the largest inter-group entropy range:")
top5 = (fam_df.nlargest(5, "entropy_range")
        [["dataset", "n_groups", "pop_entropy", "mean_group_entropy", "entropy_range"]]
        .reset_index(drop=True)
        .round(3))
display(top5)

**What the pop/group contrast reveals:**

- For most families, **group entropies cluster near the population entropy** — points sit close
  to the diagonal. Groups often share the population's broad shape; conditioning changes *which*
  options carry the mass more often than it changes *how spread out* the mass is.
- The minority of families that scatter far off-diagonal are the genuinely
  **demographically polarized** items: one segment concentrates on one answer, another spreads
  across several. These are the richest signal for the grouped split and the hardest cases for
  a model.
- **Afrobarometer and ESS** show the highest mean inter-group entropy range — cross-national
  and cross-regional items where political, religious, and cultural divides produce real
  distributional shifts between segments.
- **LatinoBarometro** and **ISSP** tend to show lower range — groups within these surveys are
  more internally homogeneous on many questions.
- **Implication for the error decomposition below:** a system that nails *location* (right
  options) but ignores *concentration* (right spread) will fail hardest on high-range families,
  because those are the items where shape — not just mode — differs across groups.

---

## Part II: Error Decomposition — Diagnosing a Specific Run

With the dataset landscape established, we now take **one run** and ask *why* its predicted
distributions are wrong. The decomposition splits each item's TVD(P, Q) into two
non-negative, interpretable parts:

```
TVD(P, Q) = concentration_err + location_err
```

- **concentration error** — mismatch in the *profile* (how peaked vs. spread the mass is),
  independent of which options carry it. Am I too diffuse or too sharp?
- **location error** — extra error from putting well-shaped mass on the *wrong* options.
  Is the weight in the right place?

Because the normalizer Z depends only on the truth, a score *gain* between two systems
decomposes the same way — letting us say "that improvement fixed the spread" or "it moved
mass onto the right options".

In [ ]:
# --- pick a run ----------------------------------------------------------------
# `best_analysis_run()` returns the faithful-vs-final-router run by default.
# Switch to "model-sweep" or "fulldev" for alternative comparisons.
from scrye.results import best_analysis_run, ANALYSIS_RUNS

RUN_KEY = "final"
results_path = best_analysis_run(RUN_KEY)
RUN = results_path.stem          # e.g. "2026-06-21-decomp-final"
per_system = json.load(open(results_path))
meta = ANALYSIS_RUNS[RUN_KEY]
print(f"run: {RUN}")
print(f"description: {meta['description']}")
print(f"best system: {meta['best_system']}")
print(f"systems ({len(per_system)}): {list(per_system.keys())}")

## Part II setup: normalizers + per-item decomposition

`Z` (SimBench Eq. 2) is built from the **full split** so score-scale columns
line up with reported scores. We then decompose every record of every system
into `concentration_err` (wrong spread) and `location_err` (wrong options).

The default run (`2026-06-21-decomp-final`) compares:
- **`faithful`** — the SimBench paper baseline prompt, no conditioning, no routing
- **`final_router`** — the Stage 17 champion: `RoutingPredictor` (task-kind
  classifier → task_context for surveys/knowledge, abstain for risky_choice/
  moral_dilemma, calibrated_commitment for personality/other)
  + `AbstainCalibrator(OSPsychMACH)`

In [ ]:
normalizers = build_normalizers(split.dev + split.val + split.test)

# one decomposed frame per system; tag with the system label
frames = {}
for sysname, recs in per_system.items():
    df = decompose_records(recs, normalizers=normalizers)
    df["system"] = sysname
    frames[sysname] = df

all_df = pd.concat(frames.values(), ignore_index=True)
print(f"decomposed {len(all_df)} (record × system) rows")
all_df[["system", "split", "dataset", "tvd", "concentration_err",
        "location_err", "entropy_gap", "mode_match", "score"]].head()

## Headline: per-system error profile

Score loss = `100 − S` split into **concentration** (wrong spread) and
**location** (wrong options). The gap between `faithful` and `final_router`
shows what the full Stage 01–16 method development actually fixed.

In [ ]:
def summary_table(df, split="grouped"):
    rows = []
    for sysname, f in df.groupby("system"):
        s = system_summary(f[f["split"] == split])
        s.name = sysname
        rows.append(s)
    return pd.DataFrame(rows)

grouped_tbl = summary_table(all_df, "grouped").sort_values("score_loss")
cols = ["n", "score_loss", "conc_loss", "loc_loss", "conc_share",
        "entropy_gap", "mode_acc", "mode_mass_err"]
display(grouped_tbl[cols].round(3))

print("\nReading:")
print("  conc_share ~0.65-0.89 -> most error is PROFILE/spread, not wrong-options")
print("  entropy_gap > 0        -> predictions are MORE diffuse than truth (too flat)")


## The improvement, decomposed

The stacked bars show how much of the `faithful → final_router` gain came from
fixing the **spread profile** (concentration) vs. placing mass on the
**right options** (location).

In [ ]:
def loss_split_bar(tbl, title):
    t = tbl.sort_values("score_loss", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(t) + 1.5))
    y = np.arange(len(t))
    ax.barh(y, t["conc_loss"], color="#4C72B0", label="concentration (wrong spread)")
    ax.barh(y, t["loc_loss"], left=t["conc_loss"], color="#DD8452",
            label="location (wrong options)")
    ax.set_yticks(y); ax.set_yticklabels(t.index)
    ax.invert_yaxis()
    ax.set_xlabel("score loss below 100  (= 100 - SimBench S)")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
    for yi, (c, l) in enumerate(zip(t["conc_loss"], t["loc_loss"])):
        ax.annotate(f"{c+l:.0f}", (c + l, yi), xytext=(3, 0),
                    textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

loss_split_bar(grouped_tbl, "Grouped score loss = concentration + location")


In [ ]:
# Attribute the faithful → final_router improvement
def get(sys_label):
    return grouped_tbl.loc[sys_label] if sys_label in grouped_tbl.index else None

pairs = [
    ("faithful → final_router", "faithful", "final_router"),
]
print(f"{'transition':46s} {'dScore':>8} {'dConc':>8} {'dLoc':>8}  attribution")
for label, a, b in pairs:
    ra, rb = get(a), get(b)
    if ra is None or rb is None:
        print(f"{label:46s}  (system not found in this run)")
        continue
    dscore = ra["score_loss"] - rb["score_loss"]     # positive = improvement
    dconc  = ra["conc_loss"]  - rb["conc_loss"]
    dloc   = ra["loc_loss"]   - rb["loc_loss"]
    drv = "concentration" if abs(dconc) > abs(dloc) else "location"
    print(f"{label:46s} {dscore:+8.2f} {dconc:+8.2f} {dloc:+8.2f}  mostly {drv}")

## Distributional comparison: predicted vs true *spread*

Each point is one item: predicted normalized entropy vs the truth's. Points
**above** the diagonal = the system is too diffuse (over-spread); **below** =
too sharp. This is the concentration error made visual — and doubles as the
entropy-calibration diagnostic (is there a systematic, fixable bias, or just
noise around the line?).


In [ ]:
def entropy_scatter(systems, split="grouped"):
    n = len(systems)
    cols = min(3, n); rows_n = int(__import__("math").ceil(n / cols))
    fig, axes = plt.subplots(rows_n, cols,
                             figsize=(4.2 * cols, 3.8 * rows_n), squeeze=False)
    for ax, sysname in zip(axes.flat, systems):
        f = frames[sysname]; f = f[f["split"] == split]
        ax.scatter(f["truth_entropy"], f["pred_entropy"], s=8, alpha=0.35,
                   color="#55A868")
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        gap = f["entropy_gap"].mean()
        ax.set_title(f"{sysname}\nmean gap {gap:+.3f}", fontsize=8)
        ax.set_xlabel("truth entropy"); ax.set_ylabel("pred entropy")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

entropy_scatter(list(per_system.keys()))

## Where do the gains land? Error by truth-entropy regime

SimBench finds models do worst on high-entropy (contested) items. Binning items
into low / mid / high truth-entropy tertiles shows whether a system's advantage
comes from the easy consensus items or the hard contested ones.


In [ ]:
def by_entropy_regime(df, systems, split="grouped"):
    g = df[(df["split"] == split) & (df["system"].isin(systems))].copy()
    # tertiles on the pooled truth-entropy distribution (shared bin edges)
    edges = g["truth_entropy"].quantile([0, 1/3, 2/3, 1.0]).to_numpy(copy=True)
    edges[0] -= 1e-9
    g["regime"] = pd.cut(g["truth_entropy"], bins=edges,
                         labels=["low (consensus)", "mid", "high (contested)"])
    piv = g.groupby(["system", "regime"], observed=True).apply(
        lambda x: 100 - system_summary(x)["score_loss"]).unstack()
    return piv

# use all systems in the run
show = list(per_system.keys())
regime_tbl = by_entropy_regime(all_df, show)
print("mean SimBench score by truth-entropy regime (grouped):")
display(regime_tbl.round(1))

## Side-by-side: predicted vs true distributions

Concrete items — the best-fit, a typical, and the worst-fit grouped item for the
chosen system — with each item's concentration/location split annotated.


In [ ]:
# Default to final_router; fall back to whatever is in this run
FOCUS = "final_router"
FOCUS = FOCUS if FOCUS in frames else list(frames)[0]
recs = per_system[FOCUS]
fdf = frames[FOCUS]
g_idx = fdf.index[fdf["split"] == "grouped"].tolist()
gsub = fdf.loc[g_idx].sort_values("tvd")
picks = {"best fit":  gsub.index[0],
         "typical":   gsub.index[len(gsub) // 2],
         "worst fit": gsub.index[-1]}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (label, idx) in zip(axes, picks.items()):
    rec = recs[idx]
    opts = list(rec["truth"])
    truth = [rec["truth"].get(o, 0) for o in opts]
    pred  = [rec["pred"].get(o, 0)  for o in opts]
    x = np.arange(len(opts)); w = 0.4
    ax.bar(x - w/2, truth, w, label="truth", color="#4C72B0")
    ax.bar(x + w/2, pred,  w, label="pred",  color="#DD8452")
    row = fdf.loc[idx]
    ax.set_title(f"{label} | {rec['dataset']}\nconc {row['concentration_err']:.2f} / "
                 f"loc {row['location_err']:.2f}", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(opts, fontsize=7); ax.legend(fontsize=7)
fig.suptitle(f"{FOCUS} — predicted vs true (grouped)", fontsize=10)
plt.tight_layout(); plt.show()

## Pop vs grouped contrast

Pop is near-unconditioned, so the *location* axis has little to grab onto there;
grouped is where option-placement matters. Contrasting the split confirms the
conditioning gains are a grouped-split phenomenon.


In [ ]:
rows = []
for split in ("pop", "grouped"):
    t = summary_table(all_df, split)
    t["split"] = split
    rows.append(t[["split", "score_loss", "conc_loss", "loc_loss", "conc_share", "entropy_gap"]])
contrast = pd.concat(rows).reset_index().rename(columns={"index": "system"})
display(contrast.round(3).sort_values(["system", "split"]).set_index(["system", "split"]))


## What this run says

*(decomp-final; grouped split; faithful vs final task-kind router)*

1. **Most error is still concentration** (`conc_share` ≈ 0.65–0.85 for both
   systems). Even after all improvements, the dominant failure is getting the
   *shape* (spread profile) wrong, not the option labels.

2. **The router's gain is split between both error types** but skews toward
   **location**: the task-kind routing + task-context prompt places mass on
   the right options more reliably, especially for opinion surveys and
   knowledge items where sibling-item context helps.

3. **We are still too diffuse** (`entropy_gap > 0`): both systems predict more
   uniform distributions than the truth. The router narrows this gap slightly
   (better calibrated_commitment prompting), but concentration error remains
   the main open headroom.

4. **Pop gains are larger than grouped** (in line with stage-17 test): the
   routing abstains on hard behavioral tasks (risky_choice, moral_dilemma)
   that hurt pop more than grouped, and task_context helps pop opinion surveys.

5. **The open frontier** is concentration error on contested (high H) items.
   These require capturing genuine spread, not just picking the right mode —
   which no current prompt strategy has cracked.